## Some Important Concept or Mathematical Operator in Signal Processing

### Impulse Response & Step Response
1. **Impulse Response (IR)**: This is the response of a system to a unit impulse input. A unit impulse is a signal that has an infinite amplitude at $t = 0$ and zero elsewhere, typically denoted as $\delta(t)$. The impulse response is usually representated as $h(t)$.

2. **Step Response (SR)**: This is the response of a system to a unit step input. A unit step signal jumps from $0$ to $1$ at $t = 0$, usually denoted as $u(t)$. The step response is typically represented as $s(t)$.

There is a mathematical relationship bewtween these two response. The impulse response and the step response are related through differentiation and integration:

* The step response is the integral of the impulse response:
$$
s(t) = \int^{t}_{-\infty} h(\tau) d\tau
$$

* Conversely, the impulse response is the derivative of the step response:
$$
h(t) = \frac{d}{dt} s(t)
$$

#### Extract Impulse Response from Step Response (Usage of the Relationship)
If you have measured the step response of a system, you can extract the impulse response by differentiating the step response.

##### Practical Steps
1. **Measure the Step Response**:
* Apply a unit step input to the system and measure the output to get the step response $s(t)$.

2. **Differentiate the Step Response**:
* Compute the derivative of the measured step response to obtain the impulse response $h(t)$.

### Convolution Operation

#### Definition
* $x(t)$: input signal,
* $h(t)$: impulse response,
* $y(t)$: output signal,
$$
y(t) = (x \ast h)(t) = \int^{\infty}_{-\infty} x(\tau) h(t - \tau) d\tau
$$

### A. Cryoscope reconstruction of a typical step response
An overview of the distortion models used can be found in the table below.


1. **HDAWG**
* The response of the HDAWG is taken into account by performing a convolution with an impulse response extracted from the measured step response.

* This step response was measured when the HDAWG was operated in ***amplified mode*** and shown in FIG. S1. (In the Supplementary of the paper.) => Measured

2. **Bias Tee**
  * The step response of the bias tee is modeled as a single exponential high-pass filter of the form $s(t) = e^{-t / \tau_{HP}} \cdot u(t)$ and serveral exponetial filters of the form $s(t) = 1 + A \cdot e^{-t / \tau_{IIR}} \cdot u(t)$,  
  where $\tau_{IIR}$ and $\tau_{HP}$ are the relevant time constants,  
  $A$ is an amplitude coefficient and $u(t)$ is the Heaviside step function.  
  
  The coefficients used are based on a measured step response for a bias tee and are the same as in Ref. S1(in the supplementary of the paper).
    
3. **Skin Effect**
  * The skin effect is modeled according to Ref. S2 with an attenuation of $\alpha_{\text{GHz}} = 2.1\; \text{dB}$ at $1\; \text{GHz}$.  
  $$
    s(t) = ( 1 - \text{erfc}(\alpha_{\text{GHz}} / 12 \sqrt{t})) \cdot u(t)
  $$

4. **On-chip Response**
  $$
  s(t) = (1 + A \cdot e^{-t / \tau_{IIR}}) \cdot u(t)
  $$

In [ ]:
"""
Import the necessary libraries (modules, packages)
"""

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.integrate import quad
import scipy.special as sp
import scipy.signal as signal
from lmfit import Minimizer, Parameters

In [ ]:
"""
Define the parameters of the problem
"""

# Parameters from the table
# For Bias Tee (Might be different)
tau_HP = 41 * 1e-6 # 41 us
# tau_IIR1, A1 = 15e-6, 0.13
# tau_IIR2, A2 = 6.4e-6, 0.99
# tau_IIR3, A3 = 2e-9, 0.6
# Constants for the skin effect
alpha_GHz = 2.1  # dB
# Constants for the on-chip response
tau_IIR_onchip = 2e-9  # 2 ns
A_onchip = 0.6

# Set the sampling rate (e.g., 1 GHz, which corresponds to 1 ns per sample)
sampling_rate = 1e10  # 1 GHz
sampling_interval = 1 / sampling_rate  # 1 ns

# Set the total measurement time or pulse duration (e.g., 1 microsecond)
total_time = 101e-9  # 1 microsecond

# Generate time vector
t = np.arange(0, total_time, sampling_interval)  # Generate time points with 1 ns interval

print(f"Sampling Interval: {sampling_interval} second.")
print(f"Sampling Rate: {sampling_rate:.2e} Hz.")

In [ ]:
"""
Define the Distortion Model of the Components
"""
# Define the heaviside function with a time shift
def heaviside(t, t0=0):
    return np.heaviside(t - t0, 1)

# Define delta function for convenience (Dirac delta approximation)
def delta_function(t, eps=1e-9, t0=0):
    return np.where(np.abs(t - t0) < eps, 1.0/eps, 0.0)

# Define step response functions
def step_response_awg(t, t0=0):
    # 定義二階系統階躍響應
    alpha = 2e9  # 衰減因子
    beta = 1e9   # 振蕩頻率
    return 1 - np.exp(-alpha * t) * (np.cos(beta * t) + (alpha / beta) * np.sin(beta * t)) # 用二階做嘗試

def step_response_bias_tee(t, tau_HP, t0=0):
    return np.exp(-t / tau_HP) * np.heaviside(t - t0, 1)

def step_response_skin(t, alpha_GHz, t0=0):
    return (1 - sp.erfc(alpha_GHz / (21 * np.sqrt(t)))) * np.heaviside(t - t0, 1)

def step_response_onchip(t, tau_IIR, A, t0=0):
    return (1 + A * np.exp(-t / tau_IIR)) * np.heaviside(t - t0, 1)

# Define impulse response functions by differentiating step responses
def impulse_response_awg(t, t0=0):
    return np.gradient(step_response_awg(t, t0), (1 / sampling_rate))

# Define impulse response functions with a shifted step function
def impulse_response_bias_tee(t, tau_HP, t0=0):
    return np.gradient(step_response_bias_tee(t, tau_HP, t0), (1 / sampling_rate))

def impulse_response_skin(t, alpha_GHz, t0=0):
    return np.gradient(step_response_skin(t, alpha_GHz, t0), (1 / sampling_rate))

def impulse_response_onchip(t, tau_IIR, A, t0=0):
    return np.gradient(step_response_onchip(t, tau_IIR, A, t0), (1 / sampling_rate))


In [ ]:
"""
Define some interval (or we might call it range?) or signal
"""
# Define control pulse
control_pulse = np.zeros_like(t) # Create an zero array as long as time interval 't'
control_pulse_amplitude = 0.01 # Volt
start_time = 10e-9 # Assume the pulse start at 10 ns
end_time = start_time + 10e-9 # Assume the pulse lasts 100 ns
control_pulse[(t >= start_time) & (t <= end_time)] = control_pulse_amplitude # Assume the amplitude is 1 V

# # Generate impulse responses for all components
# impulse_responses = [
#     # impulse_response_awg(t),
#     impulse_response_bias_tee(t, tau_HP, t0=start_time),
#     impulse_response_skin(t, alpha_GHz, t0=start_time),
#     # impulse_response_onchip(t, tau_IIR1, A1),
#     # impulse_response_onchip(t, tau_IIR2, A2),
#     # impulse_response_onchip(t, tau_IIR3, A3),
# ]

In [ ]:
"""
Do the signal processing
"""
# Convolve control pulse with each impulse response
signal_out = control_pulse.copy()

signal_out_after_awg = np.convolve(signal_out, impulse_response_awg(t, t0 = start_time), mode='full')[:len(t)] * (t[1] - t[0])
signal_out_after_bias_tee = np.convolve(signal_out_after_awg, impulse_response_bias_tee(t, tau_HP, t0 = start_time), mode='full')[:len(t)] * (t[1] - t[0])
signal_out_after_skin_effect = np.convolve(signal_out_after_bias_tee, impulse_response_skin(t, alpha_GHz, t0 = start_time), mode='full')[:len(t)] * (t[1] - t[0])
signal_out_after_onchip = np.convolve(signal_out_after_skin_effect, impulse_response_onchip(t, tau_IIR_onchip, A_onchip, t0 = start_time), mode='full')[:len(t)] * (t[1] - t[0])

In [ ]:
"""
Define the function of Filters
"""
# Define FIR filter based on the description in the paper
def fir_filter(signal_in, b):
    """
    Apply the FIR filter to the input signal using the calculated coefficients.
    
    Args:
    signal_in (array): Input signal to be filtered.
    b (array): Coefficients of the FIR filter.
    
    Returns:
    signal_out (array): Filtered signal.
    """
    y = np.zeros_like(signal_in)
    N = len(b)
    for n in range(len(signal_in)):
        for i in range(N):
            if n - i >= 0:
                y[n] += b[i] * signal_in[n - i]
    return y

# Define IIR filter based on the description in the paper
def iir_filter_params(tau_IIR, A, fs):
    """
    Calculate the coefficients of a first-order IIR filter that corrects for an exponential over- or undershoot.
    
    Args:
    tau_IIR (float): Time constant of the IIR filter.
    A (float): Amplitude of the exponential term.
    fs (float): Sampling rate.
    
    Returns:
    b (list): Numerator coefficients of the IIR filter.
    a (list): Denominator coefficients of the IIR filter.
    """
    alpha = 1 - np.exp(-1 / (fs * tau_IIR * (1 + A)))
    
    if A < 0:
        k = A / ((1 + A) * (1 - alpha))
    else:
        k = A / (1 + A - alpha)
    
    b0 = 1 - k + k * alpha
    b1 = -(1 - k) * (1 - alpha)
    a0 = 1
    a1 = -(1 - alpha)
    
    b = [b0, b1]
    a = [a0, a1]
    
    return b, a

# IIR filter function using the coefficients calculated from iir_filter_params
def iir_filter(signal_in, b, a):
    """
    Apply the IIR filter to the input signal using the calculated coefficients.
    
    Args:
    signal_in (array): Input signal to be filtered.
    b (array): Numerator coefficients of the IIR filter.
    a (array): Denominator coefficients of the IIR filter.
    
    Returns:
    signal_out (array): Filtered signal.
    """
    y = np.zeros_like(signal_in)
    for n in range(len(signal_in)):
        y[n] = b[0] * signal_in[n]
        if n > 0:
            y[n] += b[1] * signal_in[n-1] - a[1] * y[n-1]
    return y

In [ ]:
"""
Define the Optimization Function for Adjusting the Parameters
"""
# Optimize FIR filter coefficients
def optimize_fir(signal_in, signal_target, n):
    """
    Optimize the FIR filter coefficients to minimize the difference between the filtered signal and the target signal.
    
    Args:
    signal_in (array): Input signal to be filtered.
    signal_target (array): Target signal for comparison.
    n (int): Number of taps for the FIR filter.
    
    Returns:
    array: Optimized FIR filter coefficients.
    """
    def cost_function(params):
        taps = np.array([params[f'coeff_{i}'] for i in range(n)])
        filtered_signal = fir_filter(signal_in, taps)
        return filtered_signal - signal_target
    
    params = Parameters()
    for i in range(n):
        params.add(f'coeff_{i}', value=0.0)

    minner = Minimizer(cost_function, params)
    result = minner.minimize()
    return np.array([result.params[f'coeff_{i}'].value for i in range(n)])


# Optimize IIR filter coefficients
def optimize_iir(signal_in, signal_target, tau_IIR, A):
    """
    Optimize the IIR filter coefficients to minimize the difference between the filtered signal and the target signal.
    
    Args:
    signal_in (array): Input signal to be filtered.
    signal_target (array): Target signal for comparison.
    tau_IIR (float): Initial time constant for the IIR filter.
    A (float): Initial amplitude for the IIR filter.
    
    Returns:
    (float, float): Optimized time constant and amplitude for the IIR filter.
    """
    def cost_function(params):
        tau_IIR = params['tau_IIR']
        A = params['A']
        b, a = iir_filter_params(tau_IIR, A, fs)
        filtered_signal = iir_filter(signal_in, b, a)
        return filtered_signal - signal_target
    
    params = Parameters()
    params.add('tau_IIR', value=tau_IIR)
    params.add('A', value=A)

    minner = Minimizer(cost_function, params)
    result = minner.minimize()
    return result.params['tau_IIR'].value, result.params['A'].value


In [ ]:
# Parameters for the FIR filter
n_taps = 40  # Number of taps for the FIR filter

fir_params = optimize_fir(signal_out, control_pulse, n_taps)

In [ ]:
# Create a plotly figure
fig = go.Figure()

fig.add_trace(go.Scatter(x=t, y=control_pulse, mode='lines', name='Control Pulse'))
fig.add_trace(go.Scatter(x=t, y=signal_out_after_awg, mode='lines', name='Output Signal - After AWG'))
fig.add_trace(go.Scatter(x=t, y=signal_out_after_bias_tee, mode='lines', name='Output Signal - After Bias Tee'))
fig.add_trace(go.Scatter(x=t, y=signal_out_after_skin_effect, mode='lines', name='Output Signal - After Skin Effect'))
fig.add_trace(go.Scatter(x=t, y=signal_out_after_onchip, mode='lines', name='Output Signal - After On-Chip'))

# Update layout
fig.update_layout(
    title='Signal Correction Using FIR and IIR Filters',
    xaxis_title='Time (s)',
    yaxis_title='Amplitude',
    legend_title='Signals'
)

# Show plot
fig.show()


In [ ]:
sampling_interval